# NLP Lab Task 5 — N-gram Language Models

**Standard library only.** Each numbered task has its own code cell. Run the cells from top to bottom.

Conventions used:
- All text is lowercased and split on spaces.
- Each sentence is represented as `['<s>', ..., '</s>']`.
- The vocabulary `V` contains all words plus `</s>`, but excludes `<s>`.
- Sentence probabilities include the end marker `</s>` and do not predict `<s>`.
- For the trigram model, the first predicted token is scored with the bigram model from `<s>`; later tokens use trigram probabilities.
- Perplexity is computed over predicted tokens (including `</s>`, excluding `<s>`).


## Task 1: Tokenization

In [1]:
from collections import Counter
from itertools import permutations
from math import log, exp, inf, isclose
from itertools import product

train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]
test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]

train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("Tokenized training sentences:")
for sentence in train_tokens:
    print(sentence)
print("\nTokenized test sentences:")
for sentence in test_tokens:
    print(sentence)

with_markers = sum(len(s) for s in train_tokens)
without_markers = sum(len(s) - 2 for s in train_tokens)
print("\nTraining tokens including markers:", with_markers)
print("Training tokens excluding markers:", without_markers)

def build_vocab(sentences):
    # Accept raw sentences or already-tokenized sentences.
    vocab = set()
    for sentence in sentences:
        tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
        vocab.update(tokens)
    vocab.discard("<s>")
    return vocab

V_set = build_vocab(train_tokens)
V = len(V_set)
print("\nV =", V)
print("Sorted vocabulary:", sorted(V_set))

Tokenized training sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Tokenized test sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Training tokens including markers: 30
Training tokens excluding markers: 22

V = 11
Sorted vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

In [2]:
def get_ngrams(tokens, n):
    if n < 1:
        raise ValueError("n must be at least 1")
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

first = train_tokens[0]
for n in (1, 2, 3):
    print(f"{n}-grams of first training sentence:")
    print(get_ngrams(first, n))

print("\nVerify m - n + 1 formula:")
for sent_i, tokens in enumerate(train_tokens, start=1):
    m = len(tokens)
    for n in (1, 2, 3):
        grams = get_ngrams(tokens, n)
        expected = max(0, m - n + 1)
        print(f"Training sentence {sent_i}: m={m}, n={n}, actual={len(grams)}, expected={expected}")
        assert len(grams) == expected

long_n = len(first) + 1
print(f"\nn={long_n} (larger than sentence length {len(first)}):", get_ngrams(first, long_n))

1-grams of first training sentence:
[('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
2-grams of first training sentence:
[('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
3-grams of first training sentence:
[('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]

Verify m - n + 1 formula:
Training sentence 1: m=8, n=1, actual=8, expected=8
Training sentence 1: m=8, n=2, actual=7, expected=7
Training sentence 1: m=8, n=3, actual=6, expected=6
Training sentence 2: m=7, n=1, actual=7, expected=7
Training sentence 2: m=7, n=2, actual=6, expected=6
Training sentence 2: m=7, n=3, actual=5, expected=5
Training sentence 3: m=8, n=1, actual=8, expected=8
Training sentence 3: m=8, n=2, actual=7, expected=7
Training sentence 3: m=8, n=3, actual=6, expected=6
Training sentence 4: m=7, n=1, actual=7, expected=7
Training senten

## Task 3: Counting N-grams

In [3]:
unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in train_tokens:
    unigram_counts.update(get_ngrams(tokens, 1))
    bigram_counts.update(get_ngrams(tokens, 2))
    trigram_counts.update(get_ngrams(tokens, 3))

def print_count_table(counter, title):
    print(title)
    for gram, count in sorted(counter.items(), key=lambda item: (-item[1], item[0])):
        print(f"{gram}: {count}")
    print()

print_count_table(unigram_counts, "Unigram counts (highest to lowest):")
print_count_table(bigram_counts, "Bigram counts (highest to lowest):")
print_count_table(trigram_counts, "Trigram counts (highest to lowest):")

print("Five most common bigrams:", bigram_counts.most_common(5))
print("Distinct unigrams:", len(unigram_counts))
print("Distinct bigrams:", len(bigram_counts))
print("Distinct trigrams:", len(trigram_counts))

def count_of(ngram):
    n = len(ngram)
    counter = {1: unigram_counts, 2: bigram_counts, 3: trigram_counts}.get(n)
    return counter.get(tuple(ngram), 0) if counter is not None else 0

seen_bigram = next(iter(bigram_counts))
unseen_bigram = ("fish", "rug")
print(f"\nSeen bigram {seen_bigram}: {count_of(seen_bigram)}")
print(f"Unseen bigram {unseen_bigram}: {count_of(unseen_bigram)}")

Unigram counts (highest to lowest):
('the',): 8
('</s>',): 4
('<s>',): 4
('ate',): 2
('cat',): 2
('dog',): 2
('on',): 2
('sat',): 2
('bone',): 1
('fish',): 1
('mat',): 1
('rug',): 1

Bigram counts (highest to lowest):
('<s>', 'the'): 4
('ate', 'the'): 2
('on', 'the'): 2
('sat', 'on'): 2
('the', 'cat'): 2
('the', 'dog'): 2
('bone', '</s>'): 1
('cat', 'ate'): 1
('cat', 'sat'): 1
('dog', 'ate'): 1
('dog', 'sat'): 1
('fish', '</s>'): 1
('mat', '</s>'): 1
('rug', '</s>'): 1
('the', 'bone'): 1
('the', 'fish'): 1
('the', 'mat'): 1
('the', 'rug'): 1

Trigram counts (highest to lowest):
('<s>', 'the', 'cat'): 2
('<s>', 'the', 'dog'): 2
('sat', 'on', 'the'): 2
('ate', 'the', 'bone'): 1
('ate', 'the', 'fish'): 1
('cat', 'ate', 'the'): 1
('cat', 'sat', 'on'): 1
('dog', 'ate', 'the'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('on', 'the', 'rug'): 1
('the', 'bone', '</s>'): 1
('the', 'cat', 'ate'): 1
('the', 'cat', 'sat'): 1
('the', 'dog', 'ate'): 1
('the', 'dog', 'sat'): 1
('the', 'fish', 

## Task 4: Unigram Model

In [4]:
N = sum(unigram_counts.values())

def unigram_prob(w):
    return unigram_counts.get((w,), 0) / N if N else 0.0

print("Unigram probabilities:")
for (word,), count in sorted(unigram_counts.items()):
    print(f"{word:>8}  count={count:<2}  P={unigram_prob(word):.4f}")

assert isclose(sum(unigram_prob(w) for (w,) in unigram_counts), 1.0, abs_tol=1e-12)
print("Sum of unigram probabilities = 1.0000")

def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
    # Exclude <s>, include </s>.
    probability = 1.0
    for word in tokens[1:]:
        probability *= unigram_prob(word)
    return probability

for sentence in test:
    print(f"P_unigram({sentence!r}) = {unigram_sentence_prob(sentence):.8f}")

words = test[0].lower().split()
permutation_results = {}
for order in permutations(words):
    ordered_sentence = " ".join(order)
    permutation_results.setdefault(unigram_sentence_prob(ordered_sentence), []).append(ordered_sentence)
print("\nUnigram probability groups for permutations of first test sentence:")
for probability, orders in sorted(permutation_results.items(), key=lambda item: item[0], reverse=True):
    print(f"P={probability:.8f}: {orders}")

Unigram probabilities:
    </s>  count=4   P=0.1333
     <s>  count=4   P=0.1333
     ate  count=2   P=0.0667
    bone  count=1   P=0.0333
     cat  count=2   P=0.0667
     dog  count=2   P=0.0667
    fish  count=1   P=0.0333
     mat  count=1   P=0.0333
      on  count=2   P=0.0667
     rug  count=1   P=0.0333
     sat  count=2   P=0.0667
     the  count=8   P=0.2667
Sum of unigram probabilities = 1.0000
P_unigram('the cat sat on the rug') = 0.00000009
P_unigram('the fish sat on the mat') = 0.00000005

Unigram probability groups for permutations of first test sentence:
P=0.00000009: ['the cat sat on the rug', 'the cat sat on rug the', 'the cat sat the on rug', 'the cat sat the rug on', 'the cat sat rug on the', 'the cat sat rug the on', 'the cat on sat the rug', 'the cat on sat rug the', 'the cat on the sat rug', 'the cat on the rug sat', 'the cat on rug sat the', 'the cat on rug the sat', 'the cat the sat on rug', 'the cat the sat rug on', 'the cat the on sat rug', 'the cat the on ru

## Task 5: Bigram Model (MLE)

In [5]:
def bigram_prob(prev, w):
    denominator = unigram_counts.get((prev,), 0)
    if denominator == 0:
        return 0.0
    return bigram_counts.get((prev, w), 0) / denominator

def next_word_distribution(prev):
    denominator = unigram_counts.get((prev,), 0)
    if denominator == 0:
        return {}
    distribution = {}
    for (p, word), count in bigram_counts.items():
        if p == prev:
            distribution[word] = count / denominator
    return distribution

dist_the = next_word_distribution("the")
print('Next-word distribution for "the":')
for word, probability in sorted(dist_the.items()):
    print(f"{word:>8}: {probability:.4f}")
assert isclose(sum(dist_the.values()), 1.0, abs_tol=1e-12)

def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
    probability = 1.0
    for prev, word in zip(tokens, tokens[1:]):
        probability *= bigram_prob(prev, word)
    return probability

for sentence in test:
    print(f"P_bigram({sentence!r}) = {bigram_sentence_prob(sentence):.8f}")

def predict_next(prev):
    distribution = next_word_distribution(prev)
    if not distribution:
        return None
    # Highest probability first; alphabetical order breaks ties.
    return sorted(distribution.items(), key=lambda item: (-item[1], item[0]))[0][0]

print("\nNext-word predictions:")
for word in ["the", "cat", "dog", "sat", "fish", "on", "ate"]:
    print(f"{word:>6} -> {predict_next(word)}")

Next-word distribution for "the":
    bone: 0.1250
     cat: 0.2500
     dog: 0.2500
    fish: 0.1250
     mat: 0.1250
     rug: 0.1250
P_bigram('the cat sat on the rug') = 0.01562500
P_bigram('the fish sat on the mat') = 0.00000000

Next-word predictions:
   the -> cat
   cat -> ate
   dog -> ate
   sat -> on
  fish -> </s>
    on -> the
   ate -> the


## Task 6: Trigram Model

In [6]:
def trigram_prob(w1, w2, w3):
    denominator = bigram_counts.get((w1, w2), 0)
    if denominator == 0:
        return 0.0
    return trigram_counts.get((w1, w2, w3), 0) / denominator

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
    if len(tokens) < 2:
        return 1.0
    probability = bigram_prob(tokens[0], tokens[1])
    for i in range(2, len(tokens)):
        probability *= trigram_prob(tokens[i-2], tokens[i-1], tokens[i])
    return probability

for sentence in test:
    print(f"P_trigram({sentence!r}) = {trigram_sentence_prob(sentence):.8f}")

print("\nDistinct n-gram counts:")
print(f"{'N-gram type':<15}{'Distinct count':>15}")
print(f"{'Unigrams':<15}{len(unigram_counts):>15}")
print(f"{'Bigrams':<15}{len(bigram_counts):>15}")
print(f"{'Trigrams':<15}{len(trigram_counts):>15}")

for raw_sentence, tokens in zip(test, test_tokens):
    sentence_bigrams = get_ngrams(tokens, 2)
    sentence_trigrams = get_ngrams(tokens, 3)
    unseen_bi = [g for g in sentence_bigrams if bigram_counts.get(g, 0) == 0]
    unseen_tri = [g for g in sentence_trigrams if trigram_counts.get(g, 0) == 0]
    print(f"\nSentence: {raw_sentence}")
    print("Unseen bigrams:", unseen_bi)
    print("Unseen trigrams:", unseen_tri)
    print("Model with more unseen N-grams:", "trigram" if len(unseen_tri) > len(unseen_bi) else "bigram" if len(unseen_bi) > len(unseen_tri) else "tie")

P_trigram('the cat sat on the rug') = 0.12500000
P_trigram('the fish sat on the mat') = 0.00000000

Distinct n-gram counts:
N-gram type     Distinct count
Unigrams                    12
Bigrams                     18
Trigrams                    19

Sentence: the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Model with more unseen N-grams: tie

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
Model with more unseen N-grams: trigram


## Task 7: Zero-Probability Problem

In [7]:
def unseen_bigrams(sentence):
    tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
    return [gram for gram in get_ngrams(tokens, 2) if bigram_counts.get(gram, 0) == 0]

for sentence in test:
    unseen = unseen_bigrams(sentence)
    probability = bigram_sentence_prob(sentence)
    print(f"Sentence: {sentence}")
    print("Unseen bigrams:", unseen)
    print(f"Bigram MLE sentence probability: {probability:.8f}")

assert any(bigram_sentence_prob(sentence) == 0 for sentence in test)

# Possible ordered word-to-word pairs among vocabulary words, excluding markers.
word_vocab = sorted(V_set - {"</s>"})
possible_pairs = [(a, b) for a in word_vocab for b in word_vocab]
unseen_word_pairs = [(a, b) for a, b in possible_pairs if bigram_counts.get((a, b), 0) == 0]
percentage = 100 * len(unseen_word_pairs) / len(possible_pairs) if possible_pairs else 0
print(f"\nUnseen word-only pairs: {len(unseen_word_pairs)} / {len(possible_pairs)}")
print(f"Percentage unseen: {percentage:.2f}%")

def safe_log(p):
    return -inf if p == 0 else log(p)

second_test_probability = bigram_sentence_prob(test[1])
print("Log probability of second test sentence:", safe_log(second_test_probability))

Sentence: the cat sat on the rug
Unseen bigrams: []
Bigram MLE sentence probability: 0.01562500
Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Bigram MLE sentence probability: 0.00000000

Unseen word-only pairs: 87 / 100
Percentage unseen: 87.00%
Log probability of second test sentence: -inf


## Task 8: Smoothing

In [8]:
def bigram_prob_addone(prev, w):
    denominator = unigram_counts.get((prev,), 0) + V
    return (bigram_counts.get((prev, w), 0) + 1) / denominator if denominator else 0.0

print("Conditional probability comparison:")
print(f"{'Probability':<22}{'MLE':>12}{'Add-one':>12}")
for prev, word in [("fish", "sat"), ("the", "cat")]:
    print(f"P({word} | {prev}){'':<9}{bigram_prob(prev, word):>12.4f}{bigram_prob_addone(prev, word):>12.4f}")

# Each previous token in V plus <s> is checked; distribution is over all words in V.
for prev in sorted((V_set - {"</s>"}) | {"<s>"}):
    smoothed_sum = sum(bigram_prob_addone(prev, word) for word in V_set)
    assert isclose(smoothed_sum, 1.0, abs_tol=1e-12), (prev, smoothed_sum)
print("\nAll add-one next-word distributions sum to 1.")

def bigram_prob_addk(prev, w, k):
    if k < 0:
        raise ValueError("k must be non-negative")
    denominator = unigram_counts.get((prev,), 0) + k * V
    numerator = bigram_counts.get((prev, w), 0) + k
    if denominator == 0:
        return 0.0
    return numerator / denominator

print("\nAdd-k probabilities for P(sat | fish):")
print(f"{'k':>8}{'Probability':>15}")
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k:>8.2f}{bigram_prob_addk('fish', 'sat', k):>15.4f}")

Conditional probability comparison:
Probability                    MLE     Add-one
P(sat | fish)               0.0000      0.0833
P(cat | the)               0.2500      0.1579

All add-one next-word distributions sum to 1.

Add-k probabilities for P(sat | fish):
       k    Probability
    1.00         0.0833
    0.50         0.0769
    0.10         0.0476
    0.01         0.0090


## Task 9: Perplexity and Model Comparison

In [9]:
def perplexity(sentences, prob_fn):
    log_probability_sum = 0.0
    token_count = 0
    for sentence in sentences:
        tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
        # Score all predicted tokens, including </s>, excluding <s>.
        predicted = tokens[1:]
        for i, word in enumerate(predicted, start=1):
            if prob_fn is unigram_addone_prob:
                probability = prob_fn(word)
            elif prob_fn is bigram_addone_sentence_token_prob:
                probability = prob_fn(tokens[i-1], word)
            elif prob_fn is bigram_mle_sentence_token_prob:
                probability = prob_fn(tokens[i-1], word)
            else:
                # Generic probability function accepts a token or (previous, token).
                try:
                    probability = prob_fn(tokens[i-1], word)
                except TypeError:
                    probability = prob_fn(word)
            if probability == 0:
                return inf
            log_probability_sum += log(probability)
            token_count += 1
    return exp(-log_probability_sum / token_count) if token_count else inf

def unigram_addone_prob(word):
    return (unigram_counts.get((word,), 0) + 1) / (N + V)

def bigram_addone_sentence_token_prob(prev, word):
    return bigram_prob_addone(prev, word)

def bigram_mle_sentence_token_prob(prev, word):
    return bigram_prob(prev, word)

def addk_sentence_token_prob_for(k):
    return lambda prev, word: bigram_prob_addk(prev, word, k)

# Evaluate the test sentences together.
results = [
    ("Bigram MLE", perplexity(test, bigram_mle_sentence_token_prob)),
    ("Bigram add-one", perplexity(test, bigram_addone_sentence_token_prob)),
    ("Unigram add-one", perplexity(test, unigram_addone_prob)),
]
print(f"{'Model':<22}{'Perplexity':>15}")
for name, value in results:
    print(f"{name:<22}{value:>15.4f}")

k_values = [1, 0.5, 0.1, 0.01]
test_k_results = [(k, perplexity(test, addk_sentence_token_prob_for(k))) for k in k_values]
print("\nAdd-k bigram perplexity on test set:")
print(f"{'k':>8}{'Perplexity':>15}")
for k, value in test_k_results:
    print(f"{k:>8.2f}{value:>15.4f}")
best_test_k, best_test_ppl = min(test_k_results, key=lambda item: item[1])
print(f"Best k directly on test set: {best_test_k} (perplexity={best_test_ppl:.4f})")

# Validation experiment: train on first three sentences, but keep V from all four.
validation_sentence = [train[3]]
validation_tokens = [tokenize(s) for s in validation_sentence]

validation_unigrams = Counter()
validation_bigrams = Counter()
for tokens in train_tokens[:3]:
    validation_unigrams.update(get_ngrams(tokens, 1))
    validation_bigrams.update(get_ngrams(tokens, 2))
validation_N = sum(validation_unigrams.values())

def validation_bigram_addk(prev, word, k):
    denominator = validation_unigrams.get((prev,), 0) + k * V
    numerator = validation_bigrams.get((prev, word), 0) + k
    if denominator == 0:
        return 0.0
    return numerator / denominator

def validation_perplexity(sentences, k):
    log_sum = 0.0
    count = 0
    for sentence in sentences:
        tokens = tokenize(sentence) if isinstance(sentence, str) else sentence
        for i in range(1, len(tokens)):
            p = validation_bigram_addk(tokens[i-1], tokens[i], k)
            if p == 0:
                return inf
            log_sum += log(p)
            count += 1
    return exp(-log_sum / count) if count else inf

validation_results = [(k, validation_perplexity(validation_sentence, k)) for k in k_values]
print("\nValidation perplexity (trained on first 3 training sentences; V from all 4):")
print(f"{'k':>8}{'Validation PPL':>18}")
for k, value in validation_results:
    print(f"{k:>8.2f}{value:>18.4f}")
best_validation_k, best_validation_ppl = min(validation_results, key=lambda item: item[1])
print(f"Best k selected on validation sentence: {best_validation_k} (validation perplexity={best_validation_ppl:.4f})")

# Refit on all four training sentences and evaluate the test set using selected k.
# The all-four counts are the original unigram_counts and bigram_counts.
validation_selected_test_ppl = perplexity(test, addk_sentence_token_prob_for(best_validation_k))
print(f"Test perplexity using validation-selected k={best_validation_k}: {validation_selected_test_ppl:.4f}")
print(f"Test perplexity using test-selected k={best_test_k}: {best_test_ppl:.4f}")
print("Comparison: validation selection is a more principled estimate of k; choosing k directly on test data is optimistic because it tunes to the test set.")

Model                      Perplexity
Bigram MLE                        inf
Bigram add-one                 5.7969
Unigram add-one               10.1247

Add-k bigram perplexity on test set:
       k     Perplexity
    1.00         5.7969
    0.50         4.5703
    0.10         2.9559
    0.01         2.6272
Best k directly on test set: 0.01 (perplexity=2.6272)

Validation perplexity (trained on first 3 training sentences; V from all 4):
       k    Validation PPL
    1.00            8.5857
    0.50            8.0163
    0.10            8.0260
    0.01           13.1304
Best k selected on validation sentence: 0.5 (validation perplexity=8.0163)
Test perplexity using validation-selected k=0.5: 4.5703
Test perplexity using test-selected k=0.01: 2.6272
Comparison: validation selection is a more principled estimate of k; choosing k directly on test data is optimistic because it tunes to the test set.


## Notes on interpreting results

- **MLE bigrams** can assign zero probability to a sentence if any transition was unseen in training.
- **Add-k smoothing** gives unseen bigrams nonzero probability for `k > 0`.
- In Task 9, the validation procedure selects `k` using only the fourth training sentence after fitting counts on the first three. The selected `k` is then evaluated on the test sentences after refitting the model on all four training sentences.
- The requested vocabulary includes `</s>` but not `<s>`. The previous-token context `<s>` is still used for sentence-initial bigrams.
